In [ ]:
# Setup repo imports (Colab-safe)
import os, sys
from importlib import import_module
from pathlib import Path


def find_repo_root():
    candidates = []
    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])

    env_root = os.environ.get("REPO_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())

    known_bases = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
        Path.home(),
    ]
    for base in known_bases:
        if base.is_dir():
            candidates.append(base)
            maybe_repo = base / "Where-You-LoRA-Matters"
            if maybe_repo.is_dir():
                candidates.append(maybe_repo)

    for c in candidates:
        src = c / "src"
        if src.is_dir():
            return c
    raise FileNotFoundError(
        "Could not find a 'src' directory. Set env REPO_ROOT=/path/to/Where-You-LoRA-Matters or update the cell manually."
    )


project_root = find_repo_root()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Prefer our local src modules over any installed packages
_datasets_mod = import_module("datasets")
if not getattr(_datasets_mod, "__file__", "").endswith("src/datasets.py"):
    raise ImportError(f"Resolved wrong datasets module: {_datasets_mod.__file__}")

from datasets import prepare_vqav2_datasets
from model_utils import setup_vl_model_and_processor
from finetuning_utils import create_lora_config_vl, train_vl_lora_with_wandb


### Hyperparameter Validation
Goal:
- Find optimal rank and LR for LLM-only LoRA
- Then use these settings for all placement strategy experiments


In [ ]:
!pip install -U "transformers>=4.45.0" accelerate bitsandbytes "torchvision" pillow datasets \
               gcsfs google-cloud-storage google-auth google-colab rich

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
import torch
import pandas as pd
from datasets import load_dataset
import random, json, math

from datasets import load_dataset
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig,AutoModelForImageTextToText
import gc
from datasets import load_dataset
from PIL import Image
import random

from typing import List, Dict
from PIL import Image
import torch, gc
import torch, gc
from typing import List, Tuple, Dict, Optional
from PIL import Image
import torch, random
from typing import List, Tuple, Dict, Optional
from PIL import Image
from datasets import Dataset
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from PIL import Image
from datasets import Image as HFImage
from datasets import load_dataset


import itertools
from datetime import datetime
import torch
import wandb
from transformers import AutoProcessor
import json

gc.collect()
torch.cuda.empty_cache()

In [ ]:
wandb.login()

In [ ]:
for name in [
    "model","processor","outputs","out","H","V","inputs","IHS","V0","Vpr",
    "image_mask","text_mask","Vproj_vec","T_fused_vec","trainer"
]:
    if name in globals(): del globals()[name]

gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())

In [ ]:
LORA_RANKS = [16, 32, 64]
LEARNING_RATES = [1e-4, 2e-4, 5e-4]

# Fixed settings for validation
model_name = "Qwen/Qwen3-VL-8B-Instruct"
min_pixels = 256*28*28
max_pixels = 512*28*28

# LLM-only target modules (standard for validation)
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Fixed LoRA settings
LORA_ALPHA_MULTIPLIER = 2  # alpha = 2 * rank
LORA_DROPOUT = 0.05

# Dataset sizes (can be smaller for validation)
TRAIN_SIZE = 5000  # Enough to see trends
VAL_SIZE = 500
TEST_SIZE = 500

# Training config
MAX_STEPS = 500  # ~1-2 epochs depending on batch size
EVAL_STEPS = 50
LOGGING_STEPS = 25
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4

In [ ]:
def generate_validation_configs():
    """Generate all combinations of rank and LR"""
    for rank, lr in itertools.product(LORA_RANKS, LEARNING_RATES):
        yield {
            "lora_r": rank,
            "lora_alpha": rank * LORA_ALPHA_MULTIPLIER,
            "learning_rate": lr,
        }

def create_validation_run_name(rank, lr, run_id):
    """Create descriptive run name"""
    return f"val_r{rank}_lr{lr:.0e}_id{run_id}"

In [ ]:
if __name__ == "__main__":
    
    validation_configs = list(generate_validation_configs())
    total_runs = len(validation_configs)
    
    print("="*70)
    print(f"HYPERPARAMETER VALIDATION: {total_runs} configurations")
    print("="*70)
    print(f"Strategy: LLM-only LoRA (attention layers)")
    print(f"Ranks: {LORA_RANKS}")
    print(f"Learning rates: {LEARNING_RATES}")
    print(f"Training samples: {TRAIN_SIZE}")
    print(f"Max steps: {MAX_STEPS}")
    print("="*70)
    
    print("\n" + "="*70)
    print("PREPARING DATASETS (once for all validation runs)")
    print("="*70)
    
    processor = AutoProcessor.from_pretrained(
        model_name,
        min_pixels=min_pixels,
        max_pixels=max_pixels
    )
    
    train_dataset, val_dataset, test_dataset = prepare_vqav2_datasets(
        processor=processor,
        train_size=TRAIN_SIZE,
        val_size=VAL_SIZE,
        test_size=TEST_SIZE,
        cache_dir=None,
    )
    
    print(f"✓ Train: {len(train_dataset)} samples")
    print(f"✓ Val: {len(val_dataset)} samples")
    print(f"✓ Test: {len(test_dataset)} samples")
    
    results = []
    

    for run_id, config in enumerate(validation_configs, 1):
        rank = config["lora_r"]
        lr = config["learning_rate"]
        alpha = config["lora_alpha"]
        
        print("\n" + "="*70)
        print(f"VALIDATION RUN {run_id}/{total_runs}")
        print("="*70)
        print(f"LoRA rank: {rank}")
        print(f"LoRA alpha: {alpha} ({LORA_ALPHA_MULTIPLIER}x)")
        print(f"Learning rate: {lr}")
        print(f"Target modules: {TARGET_MODULES}")
        
        try:
            lora_config = create_lora_config_vl(
                lora_r=rank,
                lora_alpha=alpha,
                lora_dropout=LORA_DROPOUT,
                target_modules=TARGET_MODULES
            )
            
            model, _, trainable_params, total_params = setup_vl_model_and_processor(
                model_name=model_name,
                lora_config=lora_config,
                min_pixels=min_pixels,
                max_pixels=max_pixels
            )
            
            print(f"✓ Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
            
            optimizer_config = setup_optimizer_scheduler(
                learning_rate=lr,
                lr_scheduler_type="cosine",
                warmup_steps=10,
                weight_decay=0.01,
            )
            
            wandb_config = {
                "experiment_type": "hyperparameter_validation",
                "placement_strategy": "llm_only",
                "model_name": model_name,
                "lora_r": rank,
                "lora_alpha": alpha,
                "lora_alpha_multiplier": LORA_ALPHA_MULTIPLIER,
                "lora_dropout": LORA_DROPOUT,
                "target_modules": TARGET_MODULES,
                "num_target_modules": len(TARGET_MODULES),
                "learning_rate": lr,
                "lr_scheduler_type": "cosine",
                "min_pixels": min_pixels,
                "max_pixels": max_pixels,
                "train_size": TRAIN_SIZE,
                "val_size": VAL_SIZE,
                "test_size": TEST_SIZE,
                "trainable_params": trainable_params,
                "trainable_percentage": 100*trainable_params/total_params,
            }
            
            run_name = create_validation_run_name(rank, lr, run_id)
            output_dir = f"./validation_outputs/{run_name}"
            
            print(f"\nStarting training: {run_name}")
            
            trainer = train_vl_lora_with_wandb(
                # Required
                model=model,
                train_dataset=train_dataset,
                val_dataset=val_dataset,
                test_dataset=test_dataset,
                processor=processor,
                
                # W&B
                wandb_project="lora-qwen3-hyperparam-search",
                wandb_run_name=run_name,
                wandb_entity=None,
                wandb_config=wandb_config,
                
                # Output
                output_dir=output_dir,
                
                # Training schedule
                max_steps=MAX_STEPS,
                eval_steps=EVAL_STEPS,
                logging_steps=LOGGING_STEPS,
                
                # Batch size
                batch_size=BATCH_SIZE,
                gradient_accumulation_steps=GRAD_ACCUM_STEPS,
                
                # Optimizer
                optimizer_config=optimizer_config,
                
                # Early stopping
                early_stopping_patience=3,
            )
            
            # Get best validation loss
            best_val_loss = min([log.get("eval_loss", float('inf')) 
                                for log in trainer.state.log_history 
                                if "eval_loss" in log])
            
            # Log success
            results.append({
                "run_id": run_id,
                "run_name": run_name,
                "status": "success",
                "lora_r": rank,
                "learning_rate": lr,
                "best_val_loss": best_val_loss,
                "trainable_params": trainable_params,
            })
            
            print(f"✓ Run {run_id} completed successfully")
            print(f"✓ Best val loss: {best_val_loss:.4f}")
            
            del model
            del trainer
            torch.cuda.empty_cache()
            
        except Exception as e:
            print(f"✗ Run {run_id} failed with error: {str(e)}")
            results.append({
                "run_id": run_id,
                "run_name": create_validation_run_name(rank, lr, run_id),
                "status": "failed",
                "lora_r": rank,
                "learning_rate": lr,
                "error": str(e),
            })
            continue
    
    print("\n" + "="*70)
    print("VALIDATION COMPLETE - RESULTS ANALYSIS")
    print("="*70)
    
    successful_runs = [r for r in results if r["status"] == "success"]
    
    if successful_runs:
        # Sort by best validation loss
        successful_runs.sort(key=lambda x: x["best_val_loss"])
        
        print("\nTop 3 configurations:")
        print("-" * 70)
        for i, result in enumerate(successful_runs[:3], 1):
            print(f"{i}. Rank={result['lora_r']}, LR={result['learning_rate']:.0e}")
            print(f"   Val Loss: {result['best_val_loss']:.4f}")
            print(f"   Trainable params: {result['trainable_params']:,}")
            print()
        
        best_config = successful_runs[0]
        print("="*70)
        print("RECOMMENDED CONFIGURATION FOR MAIN EXPERIMENTS:")
        print("="*70)
        print(f"LoRA Rank: {best_config['lora_r']}")
        print(f"LoRA Alpha: {best_config['lora_r'] * LORA_ALPHA_MULTIPLIER}")
        print(f"Learning Rate: {best_config['learning_rate']:.0e}")
        print(f"Best Val Loss: {best_config['best_val_loss']:.4f}")
        print("="*70)
        
        recommendation = {
            "lora_r": best_config['lora_r'],
            "lora_alpha": best_config['lora_r'] * LORA_ALPHA_MULTIPLIER,
            "learning_rate": best_config['learning_rate'],
            "best_val_loss": best_config['best_val_loss'],
            "lora_dropout": LORA_DROPOUT,
            "validation_date": datetime.now().isoformat(),
        }
        
        with open("recommended_hyperparameters.json", "w") as f:
            json.dump(recommendation, f, indent=2)
        
        print(f"\n✓ Saved to: recommended_hyperparameters.json")
        
    else:
        print("\n✗ No successful runs to analyze")
    
    # Report failures
    failed_runs = [r for r in results if r["status"] == "failed"]
    if failed_runs:
        print("\nFailed configurations:")
        for result in failed_runs:
            print(f"  - Rank={result['lora_r']}, LR={result['learning_rate']:.0e}")
            print(f"    Error: {result.get('error', 'Unknown')}")
    
    print(f"\n✓ Check W&B dashboard: wandb.ai")
    print(f"✓ Total runs: {len(results)} ({len(successful_runs)} successful)")
    print("="*70)